# <b>The Skeleton Run

---

---

# 1. The Setup

## 1. Importing Libraries

In [62]:
import os
import torch
import torch.nn as nn

import pymupdf
import re
import fitz
import pandas as pd

from transformers import(
    AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification,
    DistilBertModel
)

from peft import PeftModel

## 2. Device Configuration

In [2]:
device = "mps"

## 3. Locate the models.

In [21]:
BASE_DIR = "../models/"

### 3.1 Locate the text-classifier

In [39]:
MIRA_DISTILBERT_LORA = f"{BASE_DIR}/mira-distilbert-lora"

### 3.2 Locate the text-classifier heads.

In [44]:
MIRA_CLASSIFIER_HEADS = f"{BASE_DIR}/mira-distilbert-lora"

### 3.3 Locate the Risk Engine.

In [46]:
MIRA_RISK_CLASSIFIER = f"{BASE_DIR}/mira_risk_engine/mira-risk-classifier.pkl"

In [49]:
MIRA_RISK_ENCODER = f"{BASE_DIR}/mira_risk_engine/mira-risk-encoder.pkl"

### 3.3 Locate the GenLLM.

In [25]:
GEN_MODEL_PATH = f"{BASE_DIR}/MIRA2_qwen2.5-3b-lora"

---

## 4. Load <b>Inspection Classifier</b> for PDF's text.

In [11]:
class MIRAClassifier(nn.Module):
    def __init__(self, num_issue, num_category, num_severity, num_recurring):
        super().__init__()

        self.distilbert = DistilBertModel.from_pretrained('distilbert-base-uncased')

        hidden_size = self.distilbert.config.hidden_size

        # sub-models
        self.issue_classifier = nn.Linear(hidden_size, num_issue)
        self.category_classifier = nn.Linear(hidden_size, num_category)
        self.severity_classifier = nn.Linear(hidden_size, num_severity)
        self.recurring_classifier = nn.Linear(hidden_size, num_recurring)

    def forward(self, input_ids, attention_mask):
        outputs = self.distilbert(
            input_ids = input_ids,
            attention_mask = attention_mask
        )

        pooled_output = outputs.last_hidden_state[:, 0]

        issue_logits = self.issue_classifier(pooled_output)
        category_logits = self.category_classifier(pooled_output)
        severity_logits = self.severity_classifier(pooled_output)
        recurring_logits = self.recurring_classifier(pooled_output)

        return {
            'issue': issue_logits,
            'category': category_logits,
            'severity': severity_logits,
            'recurring': recurring_logits
        }

In [10]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

In [12]:
num_issue = 10
num_category = 7
num_severity = 4
num_recurring = 2

In [38]:
inspection_classifier = MIRAClassifier(
    num_issue = num_issue,
    num_category = num_category,
    num_severity = num_severity,
    num_recurring = num_recurring
)

In [42]:
inspection_classifier.distilbert = PeftModel.from_pretrained(
    inspection_classifier.distilbert,
    MIRA_DISTILBERT_LORA
)

In [73]:
inspection_classifier = inspection_classifier.to(device)

### 4.1 Load Classification Heads

In [43]:
heads = torch.load(
    os.path.join(
        MIRA_CLASSIFIER_HEADS,
        "classification_heads.pth"
    ),
    map_location=device
)

inspection_classifier.issue_classifier.load_state_dict(
    heads["issue_classifier"]
)

inspection_classifier.category_classifier.load_state_dict(
    heads["category_classifier"]
)

inspection_classifier.severity_classifier.load_state_dict(
    heads["severity_classifier"]
)

inspection_classifier.recurring_classifier.load_state_dict(
    heads["recurring_classifier"]
)

inspection_classifier = inspection_classifier.to(device)
inspection_classifier.eval()

MIRAClassifier(
  (distilbert): PeftModelForFeatureExtraction(
    (base_model): LoraModel(
      (model): PeftModelForFeatureExtraction(
        (base_model): LoraModel(
          (model): DistilBertModel(
            (embeddings): Embeddings(
              (word_embeddings): Embedding(30522, 768, padding_idx=0)
              (position_embeddings): Embedding(512, 768)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (transformer): Transformer(
              (layer): ModuleList(
                (0-5): 6 x TransformerBlock(
                  (attention): MultiHeadSelfAttention(
                    (dropout): Dropout(p=0.1, inplace=False)
                    (q_lin): lora.Linear(
                      (base_layer): Linear(in_features=768, out_features=768, bias=True)
                      (lora_dropout): ModuleDict(
                        (default): Dropout(p=0.1, 

### 4.2 Load Label Mappings

In [ ]:
import json

with open(
    os.path.join(
        MIRA_DISTILBERT_LORA,
        'label_mappings.json'
    ),
    'r'
) as f:
    label_mappings = json.load(f)

---

## 5. Load <b>Risk Engine

In [51]:
import joblib

risk_model = joblib.load(MIRA_RISK_CLASSIFIER)
risk_encoder = joblib.load(MIRA_RISK_ENCODER)

---

## 6. Load <b>GenLLM</b>

In [53]:
BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"
LORA_PATH = GEN_MODEL_PATH

In [64]:
genLLM_tokenizer = AutoTokenizer.from_pretrained(LORA_PATH)

In [75]:
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype = torch.float16
).to(device)

genLLM_model = PeftModel.from_pretrained(
    base_model,
    LORA_PATH
)

genLLM_model = genLLM_model.to(device)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

---

---

# HELPERs

---

### PDF Processing

In [76]:
PDF_PATH = "AI/Knowledge Bases/Coal_Mine_Inspection_Report_Concise"

Extract Text from PDF

In [77]:
def extract_pdf_text(pdf_path):

    document = fitz.open(
        pdf_path
    )

    text = ""

    for page in document:
        text += page.get_text()

    document.close()

    return text

In [78]:
def extract_findings(text):

    pattern = r'Finding\s+(F-\d+):\s*(.*?)(?=Finding\s+F-\d+:|(?:\n|\s)5\.\s*INSPECTOR[\'’]?S\s+REMARKS|$)'

    matches = re.findall(
        pattern,
        text,
        flags=re.DOTALL | re.IGNORECASE
    )

    findings = []

    for finding_id, finding_text in matches:

        findings.append({
            'finding_id': finding_id,
            'finding_text': finding_text.strip()
        })

    return findings

### Findings' Classification

In [79]:
def classify_finding(text):
    inputs = tokenizer(
        text, 
        return_tensors = 'pt',
        truncation = True,
        padding = True
    )

    inputs = {
        k: v.to(device)
        for k, v in inputs.items()
    }

    with torch.no_grad():
        outputs = inspection_classifier(
            input_ids = inputs['input_ids'],
            attention_mask = inputs['attention_mask']
        )

    issue_id = torch.argmax(
        outputs['issue'],
        dim=1
    ).item()

    category_id = torch.argmax(
        outputs['category'],
        dim=1
    ).item()

    severity_id = torch.argmax(
        outputs['severity'],
        dim=1
    ).item()

    recurring_id = torch.argmax(
        outputs['recurring'],
        dim=1
    ).item()

    return {
        'issue': label_mappings['issue'][issue_id],
        'category': label_mappings['category'][category_id],
        'severity': label_mappings['severity'][severity_id],
        'recurring': label_mappings['recurring'][recurring_id]
    }

In [80]:
def classify_findings(findings):
    results = []

    for finding in findings:
        prediction = classify_finding(finding['finding_text'])

        results.append({
                'finding_id': finding['finding_id'],
                'finding_text': finding['finding_text'],
                'issue': prediction['issue'],
                'category': prediction['category'],
                'severity': prediction['severity'],
                'recurring': prediction['recurring']
            })

    return results

### Predict Risk for all findings

In [81]:
def predict_risks(results):

    risk_results = []

    for result in results:

        risk_input = pd.DataFrame([{
            "issue": result["issue"],
            "category": result["category"],
            "severity": result["severity"],
            "recurring": result["recurring"]
        }])

        risk_input_encoded = risk_encoder.transform(
            risk_input
        )

        predicted_risk = risk_model.predict(
            risk_input_encoded
        )[0]

        probabilities = risk_model.predict_proba(
            risk_input_encoded
        )[0]

        predicted_index = list(
            risk_model.classes_
        ).index(predicted_risk)

        risk_confidence = (
            probabilities[predicted_index] * 100
        )

        risk_results.append({
            "finding_id": result["finding_id"],
            "finding_text": result["finding_text"],
            "issue": result["issue"],
            "category": result["category"],
            "severity": result["severity"],
            "recurring": result["recurring"],
            "risk_level": predicted_risk,
            "risk_confidence": round(
                risk_confidence,
                2
            )
        })

    return risk_results

### Text Generator Function

In [82]:
from transformers import TextIteratorStreamer
from threading import Thread

In [92]:
def generate_response(user_prompt, max_new_tokens=300):

    genLLM_model.eval()

    messages = [
        {
            "role": "system",
            "content": (
                "You are MIRA (Mine Intelligence and Risk Assessment), "
                "an AI-based coal mine inspection and compliance assistant. "

                "Use only the supplied inspection findings, structured AI "
                "assessment, risk information, and retrieved regulatory guidance "
                "to answer the user's question. "

                "Do not calculate, modify, or override the provided issue, "
                "category, severity, recurring status, risk level, or risk confidence. "

                "Do not invent regulations, rule numbers, legal provisions, "
                "inspection history, or evidence that is not present in the context. "

                "For compliance-related questions, use only the Retrieved Guidance "
                "provided in the user prompt. "

                "If Language is English, respond in clear professional English. "

                "If Language is Hindi, understand Romanized Hindi or Hinglish "
                "input and respond ONLY in standard Hindi using Devanagari script. "
                "Do not respond in Romanized Hindi or English."
            )
        },
        {
            "role": "user",
            "content": user_prompt
        }
    ]

    prompt = genLLM_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = genLLM_tokenizer(
        prompt,
        return_tensors="pt"
    ).to(genLLM_model.device)

    streamer = TextIteratorStreamer(
        genLLM_tokenizer,
        skip_prompt=True,
        skip_special_tokens=True
    )

    generation_kwargs = {
        **inputs,
        "streamer": streamer,
        "max_new_tokens": max_new_tokens,
        "do_sample": False,
        "pad_token_id": genLLM_tokenizer.pad_token_id,
        "eos_token_id": genLLM_tokenizer.eos_token_id
    }

    thread = Thread(
        target = genLLM_model.generate,
        kwargs = generation_kwargs
    )

    thread.start()

    response = ""

    for new_text in streamer:
        print(new_text, end="", flush=True)
        response += new_text

    thread.join()

### Prompt Builder 

In [93]:
def build_genllm_prompt(
    user_query,
    language,
    risk_results,
    retrieved_guidance
):

    findings_context = ""

    for result in risk_results:

        findings_context += f"""
Finding ID: {result['finding_id']}

Finding:

{result['finding_text']}

Issue: {result['issue']}
Category: {result['category']}
Severity: {result['severity']}
Recurring: {result['recurring']}
Risk Level: {result['risk_level']}
Risk Confidence: {result['risk_confidence']}%

"""

    return f"""Language: {language}

User Query: {user_query}

Inspection Findings:

{findings_context}

Retrieved Guidance:

{retrieved_guidance}
"""

----

----

In [94]:
pdf_path = "../Knowledge Bases/Coal_Mine_Inspection_Report_Concise.pdf"

# 1. Extract text from PDF
pdf_text = extract_pdf_text(
    pdf_path
)

# 2. Extract findings
findings = extract_findings(
    pdf_text
)

print(
    "Number of findings:",
    len(findings)
)

# 3. Classify all findings
results = classify_findings(
    findings
)

# 4. Predict risk for every finding
risk_results = predict_risks(
    results
)

print("\n--- CLASSIFICATION AND RISK RESULTS ---\n")

for result in risk_results:
    print(result)

Number of findings: 6

--- CLASSIFICATION AND RISK RESULTS ---

{'finding_id': 'F-01', 'finding_text': 'During inspection of the eastern working panel at Level -2, inadequate air circulation was observed in areas remote from the\nmain ventilation inlet. Real-time airflow measurements showed 1.9 m/s against a minimum requirement of 2.5 m/s. The\nventilation ducting in this section exhibits visible wear and multiple seams. This deficiency has been recorded in the\ninspection report from the preceding inspection period, indicating that earlier recommendations for duct replacement have not\nbeen implemented.', 'issue': 'Fire Safety', 'category': 'Mine Safety', 'severity': 'Medium', 'recurring': 'No', 'risk_level': 1, 'risk_confidence': 92.69}
{'finding_id': 'F-02', 'finding_text': 'Examination of the roof support system in the active working panel revealed significant deterioration of installed rock bolts.\nStructural integrity tests indicated that approximately 12 out of 60 bolts inspecte

In [95]:
retrieved_guidance = """
Supplied CMR 2017 guidance states that adequate ventilation must be maintained
in working areas to ensure safe working conditions. Ventilation arrangements
and equipment should be maintained in effective working condition, and adequate
airflow should be provided to prevent accumulation of harmful gases and ensure
proper circulation of air. Any identified deficiency in ventilation should be
addressed through appropriate corrective measures and verified during follow-up
inspection.
"""

In [103]:
user_prompt = build_genllm_prompt(
    user_query="Which are the most serious findings in the inspection, and why should they be prioritized?",
    language="English",
    risk_results=risk_results,
    retrieved_guidance=retrieved_guidance
)

In [104]:
generate_response(user_prompt)

The most serious findings in the inspection are F-02 and F-01. 

F-02 is categorized under Mine Safety with a Severity of High and a Risk Level of 1. The finding involves significant deterioration of installed rock bolts in the roof support system, which poses a critical stability concern. This could lead to potential hazards if not addressed promptly.

F-01 is also categorized under Mine Safety with a Severity of Medium and a Risk Level of 1. The issue here is inadequate air circulation in the ventilation ducting, which violates the requirement for adequate airflow to prevent gas accumulation and maintain proper air circulation. This deficiency has been recorded in previous inspections but has not been corrected, indicating a recurring problem.

Both of these findings should be prioritized due to their high severity and risk levels, as well as their recurrence. Addressing these issues will significantly improve the overall safety and reliability of the mine environment.